# SkyGuard — Full Pipeline (single self-contained notebook)

This notebook is the **one deliverable notebook** the brief asks for: it runs
top to bottom from nothing but the `data/` folder. All of the project's
`src/` helper code is **inlined below** (Section 0) rather than imported, so
this single file has zero dependencies outside the data and the standard
ML libraries (`numpy`, `pandas`, `lightgbm`, `scikit-learn`, `scipy`,
`sentence-transformers`). It doesn't read anything cached by another
notebook run -- every feature panel, model, and prediction here is built
fresh in this run.

It walks the same five stages as the rest of the project:

1. **Per-modality baselines** -- tabular / signal / text alone, so there's a
   number to beat before fusing.
2. **Missingness handling** -- sensor-dropout and empty-note handling, plus a
   modality-dropout training augmentation, with an ablation proving what it
   does and doesn't fix.
3. **Mid-level fusion** -- one LightGBM model on tabular+signal+text concatenated.
4. **Multi-task auxiliary head** -- `failure_mode` as a 6-class joint target,
   compared against plain fusion on the same validation folds (kept as a
   documented comparison; see Section 5 for why it isn't the final model).
5. **Calibration** -- Isotonic vs. Platt scaling on out-of-fold predictions,
   then the final calibrated test predictions written to `submission.csv`.

All cross-validation throughout is 5-fold `StratifiedGroupKFold` on
`drone_id`, so no aircraft's flights ever split across train/validation.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.stats import skew, kurtosis
from scipy.sparse import hstack
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import average_precision_score, brier_score_loss
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import IsotonicRegression

pd.set_option("display.max_columns", 50)
RANDOM_STATE = 42

LGB_PARAMS = dict(
    objective="binary",
    n_estimators=400,
    learning_rate=0.03,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    verbosity=-1,
)


## 0. Source code (inlined from `src/`)

Everything below is the project's `src/` package, copied in verbatim so this
notebook has no import-time dependency on it. Each subsection corresponds to
one module; the docstrings explain the *why* behind each design choice.


### 0.1 Data loading (`src/data.py`)

Loads the three modalities and aligns them on **tabular row order** -- the
tabular CSV is treated as the canonical ordering, and notes/signals are
reindexed onto it, so `tab.iloc[i]`, `notes.iloc[i]`, and `signals[i]` always
describe the same flight everywhere downstream.

Category universes for the four categorical columns are hardcoded from
`data_dictionary.md` rather than scanned from the files. `model` includes
`"E"`, which has **zero training rows but ~31% of test rows** -- if
categories were inferred from train alone, LightGBM would have no stable
code for `"E"` at predict time. Declaring the full universe up front keeps
category-to-code mappings identical across train and test.


In [2]:
ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data"

CATEGORY_LEVELS = {
    "model": ["A", "B", "C", "D", "E"],
    "motor_type": ["M1", "M2", "M3"],
    "firmware_version": ["v3.1", "v3.2", "v4.0"],
    "operator_region": ["north", "south", "east", "west"],
}


def load_tabular(split: str) -> pd.DataFrame:
    df = pd.read_csv(DATA_DIR / split / f"{split}_tabular.csv")
    for col, levels in CATEGORY_LEVELS.items():
        df[col] = pd.Categorical(df[col], categories=levels)
    return df


def load_notes(split: str, flight_ids: pd.Series) -> pd.DataFrame:
    notes = pd.read_csv(DATA_DIR / split / f"{split}_notes.csv")
    notes = notes.set_index("flight_id").reindex(flight_ids).reset_index()
    notes["maintenance_note"] = notes["maintenance_note"].fillna("").str.strip()
    return notes


def load_signals(split: str, flight_ids: pd.Series) -> tuple[np.ndarray, np.ndarray]:
    """Returns (signals, channel_names) with `signals` reindexed to flight_ids order."""
    z = np.load(DATA_DIR / split / f"{split}_signals.npz", allow_pickle=True)
    pos = pd.Series(np.arange(len(z["flight_id"])), index=z["flight_id"])
    signals = z["signals"][pos.loc[flight_ids].values]
    return signals, z["channel_names"]


def load_split(split: str) -> dict:
    """One call to get everything for a split, aligned on tabular row order."""
    tab = load_tabular(split)
    notes = load_notes(split, tab["flight_id"])
    signals, channel_names = load_signals(split, tab["flight_id"])
    return {"tab": tab, "notes": notes, "signals": signals, "channel_names": channel_names}


### 0.2 Cross-validation (`src/cv.py`)

Plain `GroupKFold` partitions groups into folds balancing *group count*, not
label rate -- with ~620 drones of very different sizes and a roughly even
split between drones that ever fail and drones that never do, that can leave
folds with noticeably different positive rates. `StratifiedGroupKFold` gives
the same group-disjointness guarantee but balances label rate across folds
too, so per-fold AUPRC is a fairer apples-to-apples comparison.


In [3]:
def get_folds(tab: pd.DataFrame, n_splits: int = 5, seed: int = 42):
    """Yields (train_idx, val_idx) arrays positional into `tab`, grouped by drone_id."""
    skf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    y_ = tab["failure_within_horizon"].values
    groups = tab["drone_id"].values
    yield from skf.split(np.zeros(len(tab)), y_, groups)


def assert_disjoint_groups(tab: pd.DataFrame, train_idx, val_idx):
    train_drones = set(tab["drone_id"].iloc[train_idx])
    val_drones = set(tab["drone_id"].iloc[val_idx])
    overlap = train_drones & val_drones
    assert not overlap, f"drone_id leaked across fold: {overlap}"


### 0.3 Metrics (`src/metrics.py`)

In [4]:
BASE_RATE = 0.125  # ~Pr(y=1); the floor a random ranker should score


def auprc(y_true, y_score) -> float:
    return average_precision_score(y_true, y_score)


def report_folds(fold_scores: list[float], label: str) -> dict:
    mean, std = float(np.mean(fold_scores)), float(np.std(fold_scores))
    print(f"{label}: fold AUPRCs = {[round(s, 4) for s in fold_scores]}")
    print(f"{label}: mean={mean:.4f}  std={std:.4f}  (floor~{BASE_RATE})")
    return {"fold_scores": fold_scores, "mean": mean, "std": std}


### 0.4 Tabular features (`src/features_tabular.py`)

Three layers, in increasing order of effort: snapshot columns as-is,
single-flight engineered ratios (degradation-rate proxies), and within-drone
rolling/trend features over the last 3 flights. The trend features exist
because of the label definition: `y=1` means "fails in the next 5 flights",
so the model needs to catch an *inflection point*, not just a high absolute
reading. All rolling features only look at the current + past flights of the
same `drone_id` (ordered by `flight_index`), so there's no leakage from the
future.


In [5]:
NUMERIC_COLS = [
    "battery_capacity_mAh", "max_payload_g", "propeller_in", "manufacture_batch",
    "flight_index", "payload_g", "ambient_temp_C", "wind_speed_ms",
    "flight_duration_min", "avg_throttle", "num_aggressive_maneuvers",
    "cumulative_flight_hours", "battery_cycles",
]
CATEGORICAL_COLS = list(CATEGORY_LEVELS.keys())
TREND_WINDOW = 3


def _rolling_slope(arr: np.ndarray) -> float:
    n = len(arr)
    if n < 2:
        return 0.0
    t = np.arange(n, dtype=np.float64)
    t -= t.mean()
    denom = (t ** 2).sum()
    return float((arr * t).sum() / denom) if denom > 0 else 0.0


def build_tabular_features(tab: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    df = tab.sort_values(["drone_id", "flight_index"]).copy()

    df["battery_cycle_rate"] = df["battery_cycles"] / df["cumulative_flight_hours"].replace(0, np.nan)
    df["payload_ratio"] = df["payload_g"] / df["max_payload_g"].replace(0, np.nan)
    df["maneuver_rate"] = df["num_aggressive_maneuvers"] / df["flight_duration_min"].replace(0, np.nan)

    grp = df.groupby("drone_id", observed=True)
    df["battery_cycles_delta"] = grp["battery_cycles"].diff().fillna(0)
    df["flight_hours_delta"] = grp["cumulative_flight_hours"].diff().fillna(0)
    df["throttle_trend"] = grp["avg_throttle"].transform(
        lambda s: s.rolling(TREND_WINDOW, min_periods=1).apply(_rolling_slope, raw=True)
    )
    df["maneuvers_roll_mean"] = grp["num_aggressive_maneuvers"].transform(
        lambda s: s.rolling(TREND_WINDOW, min_periods=1).mean()
    )
    df["payload_roll_std"] = grp["payload_g"].transform(
        lambda s: s.rolling(TREND_WINDOW, min_periods=1).std()
    ).fillna(0)

    engineered_cols = [
        "battery_cycle_rate", "payload_ratio", "maneuver_rate",
        "battery_cycles_delta", "flight_hours_delta", "throttle_trend",
        "maneuvers_roll_mean", "payload_roll_std",
    ]
    feature_cols = NUMERIC_COLS + CATEGORICAL_COLS + engineered_cols
    out = df[["flight_id"] + feature_cols]
    # restore the caller's original row order (we sorted by drone_id/flight_index
    # above purely so the rolling-window features see chronological order within
    # each drone) so positional indices -- e.g. from get_folds(tab) -- still line
    # up between `tab` and this feature frame.
    out = out.set_index("flight_id").loc[tab["flight_id"].values].reset_index()
    return out, CATEGORICAL_COLS


### 0.5 Signal features (`src/features_signal.py`)

Per channel: dropout handling, time-domain stats, and full-spectrum frequency
features. "Full-spectrum" is deliberate -- this project's EDA showed
bearing-mode flights have *elevated low-frequency* vibration power and higher
overall variance, not the high-frequency shift the naive mechanical-wear
story suggests. Hand-picking one band would have baked in the wrong prior;
binning the whole spectrum and letting the model pick the band is more
robust, and generalizes better to airframe model `"E"`, which has no
training examples to calibrate a hand-picked threshold against.

Dropout: a channel is flagged if >10% of its timesteps are exact zero. Those
spans are linearly interpolated from the channel's own non-dropout timesteps
before computing any stat -- otherwise a "missing" reading and a genuine
"voltage=0" catastrophic reading would look identical to every downstream
feature.


In [6]:
DROPOUT_FRAC_THRESHOLD = 0.10
N_BANDS = 3


def _impute_dropout(sig: np.ndarray, zero_mask: np.ndarray) -> np.ndarray:
    """sig: (N, T) for one channel. Interpolates zeroed spans per-row from that row's own data."""
    out = sig.copy()
    t = np.arange(sig.shape[1])
    for i in np.where(zero_mask.any(axis=1))[0]:
        known = ~zero_mask[i]
        if known.sum() >= 2:
            out[i, ~known] = np.interp(t[~known], t[known], sig[i, known])
        # if the whole row is zero, leave as-is: there's nothing to interpolate from
    return out


def _spectral_features(sig: np.ndarray) -> dict[str, np.ndarray]:
    """sig: (N, T) for one channel, already imputed."""
    fft = np.fft.rfft(sig, axis=1)
    power = np.abs(fft) ** 2
    power = power[:, 1:]  # drop DC bin
    total = power.sum(axis=1) + 1e-9

    n_bins = power.shape[1]
    edges = np.linspace(0, n_bins, N_BANDS + 1, dtype=int)
    band_energy = {}
    for b in range(N_BANDS):
        lo, hi = edges[b], edges[b + 1]
        band_energy[f"band{b}_energy"] = np.log1p(power[:, lo:hi].sum(axis=1))

    dominant_freq = power.argmax(axis=1).astype(np.float64)
    p = power / total[:, None]
    entropy = -(p * np.log(p + 1e-12)).sum(axis=1)

    return {
        **band_energy,
        "dominant_freq": dominant_freq,
        "spectral_entropy": entropy,
        "total_power": np.log1p(total),
    }


def build_signal_features(signals: np.ndarray, channel_names: np.ndarray) -> pd.DataFrame:
    n = signals.shape[0]
    feats = {}

    imputed = {}
    for c, name in enumerate(channel_names):
        chan = signals[:, :, c]
        zero_mask = chan == 0
        frac_zero = zero_mask.mean(axis=1)
        feats[f"{name}_frac_zero"] = frac_zero
        feats[f"{name}_is_dropout"] = (frac_zero > DROPOUT_FRAC_THRESHOLD).astype(np.float64)

        clean = _impute_dropout(chan, zero_mask)
        imputed[name] = clean

        feats[f"{name}_mean"] = clean.mean(axis=1)
        feats[f"{name}_std"] = clean.std(axis=1)
        feats[f"{name}_skew"] = skew(clean, axis=1)
        feats[f"{name}_kurtosis"] = kurtosis(clean, axis=1)
        feats[f"{name}_min"] = clean.min(axis=1)
        feats[f"{name}_max"] = clean.max(axis=1)
        feats[f"{name}_range"] = feats[f"{name}_max"] - feats[f"{name}_min"]

        t = np.arange(clean.shape[1], dtype=np.float64)
        t -= t.mean()
        denom = (t ** 2).sum()
        feats[f"{name}_slope"] = (clean * t).sum(axis=1) / denom

        for k, v in _spectral_features(clean).items():
            feats[f"{name}_{k}"] = v

    # cross-channel: motor current imbalance between the two motors
    if "motor_current_1" in imputed and "motor_current_2" in imputed:
        diff = imputed["motor_current_1"] - imputed["motor_current_2"]
        feats["current_imbalance_mean"] = diff.mean(axis=1)
        feats["current_imbalance_std"] = diff.std(axis=1)

    return pd.DataFrame(feats, index=np.arange(n))


### 0.6 Text features (`src/features_text.py`)

Two encoders to compare, plus a missingness indicator that's independent of
which encoder is used (~28% of notes are empty, and the empty rate is near
the base rate, so it's a weak feature alone, but cheap and robust). TF-IDF is
fit inside each CV fold (separate train/val text lists, fit only on train) to
keep the fold honest. Sentence embeddings come from a frozen pretrained
model, so there's nothing to fit and no leakage risk -- they're computed once
for the whole dataset.


In [7]:
_ST_MODEL = None  # lazy-loaded singleton, sentence-transformers model load is slow


def missing_indicator(texts) -> np.ndarray:
    return np.array([1.0 if t.strip() == "" else 0.0 for t in texts])


def tfidf_features(train_texts, val_texts, max_features: int = 200):
    vec = TfidfVectorizer(max_features=max_features, min_df=1)
    train_X = vec.fit_transform(train_texts)
    val_X = vec.transform(val_texts)
    return train_X, val_X


def embed_notes(texts, model_name: str = "BAAI/bge-large-en-v1.5") -> np.ndarray:
    global _ST_MODEL
    if _ST_MODEL is None:
        from sentence_transformers import SentenceTransformer
        _ST_MODEL = SentenceTransformer(model_name)
    return _ST_MODEL.encode(list(texts), show_progress_bar=False, convert_to_numpy=True)


### 0.7 Combined feature panel (`src/feature_panel.py`)

One row per `flight_id`, tabular + signal + text-embedding columns side by
side, plus a `text_missing` indicator. Text is represented by frozen BGE
embeddings rather than TF-IDF here -- they tied on the per-modality baseline,
but embeddings give a fixed-size dense vector that's trivial to concatenate
with the other two modalities and trivial to "blank out" consistently (the
empty string's own embedding), which matters for the modality-dropout work
in Stage 2.


In [8]:
def _text_embedding_frame(note_text: pd.Series) -> pd.DataFrame:
    unique_texts = note_text.unique()
    unique_emb = embed_notes(unique_texts.tolist())
    emb_lookup = dict(zip(unique_texts, unique_emb))
    emb = np.vstack(note_text.map(emb_lookup).values)
    cols = [f"text_emb_{i}" for i in range(emb.shape[1])]
    out = pd.DataFrame(emb, columns=cols, index=note_text.index)
    out["text_missing"] = missing_indicator(note_text).astype(np.float64)
    return out, cols


def build_feature_panel(tab: pd.DataFrame, note_text: pd.Series, signals: np.ndarray, channel_names: np.ndarray):
    """Returns (panel, categorical_cols, text_embedding_cols).

    `tab`, `note_text`, and `signals` must all be row-aligned (same order,
    same length) -- exactly what `load_split` already guarantees.
    """
    X_tab, cat_cols = build_tabular_features(tab)
    X_sig = build_signal_features(signals, channel_names)
    X_text, text_cols = _text_embedding_frame(note_text.reset_index(drop=True))

    panel = pd.concat(
        [X_tab.reset_index(drop=True), X_sig.reset_index(drop=True), X_text.reset_index(drop=True)],
        axis=1,
    )
    return panel, cat_cols, text_cols


### 0.8 Modality dropout (`src/modality_dropout.py`)

Only `signal` and `text` are droppable -- tabular telemetry has zero
missingness anywhere in this dataset, so there's no real-world condition
synthetic tabular dropout would simulate. Dropout is implemented at the
**raw** level (zero the `(128,8)` tensor; blank the note string) and then fed
back through the normal feature builders, rather than hand-faking feature
values -- that guarantees a synthetically-dropped row produces *exactly* the
same representation a naturally-missing one already does (e.g.
`vibration_is_dropout=1`, or the embedding of the empty string), with no
second "missing" convention to invent or get inconsistent with the first.


In [9]:
def sample_dropout_indices(idx: np.ndarray, p: float, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    mask = rng.random(len(idx)) < p
    return idx[mask]


def apply_modality_dropout(
    signals: np.ndarray,
    note_text: pd.Series,
    train_idx: np.ndarray,
    p_signal: float = 0.15,
    p_text: float = 0.15,
    seed: int = 42,
):
    """Returns (signals_aug, note_text_aug, signal_dropped_idx, text_dropped_idx).

    Only rows in `train_idx` are eligible to be dropped -- validation/test
    rows are always passed through untouched, since dropout simulates a
    training-time augmentation, not a real test-time condition.
    """
    signal_dropped_idx = sample_dropout_indices(train_idx, p_signal, seed)
    text_dropped_idx = sample_dropout_indices(train_idx, p_text, seed + 1)

    signals_aug = signals.copy()
    signals_aug[signal_dropped_idx] = 0.0

    note_text_aug = note_text.copy()
    note_text_aug.iloc[text_dropped_idx] = ""

    return signals_aug, note_text_aug, signal_dropped_idx, text_dropped_idx


## 1. Load data and build CV folds

In [10]:
train = load_split("train")
tab, notes, signals, channel_names = train["tab"], train["notes"], train["signals"], train["channel_names"]
note_text = notes["maintenance_note"]
y = tab["failure_within_horizon"].values
m = tab["failure_mode"].astype(str).str.strip().values
print(f"Train: {len(tab)} flights, {tab['drone_id'].nunique()} drones, base rate = {y.mean():.4f}")

test = load_split("test")
tab_test, notes_test, signals_test = test["tab"], test["notes"], test["signals"]
note_text_test = notes_test["maintenance_note"]
print(f"Test: {len(tab_test)} flights, {tab_test['drone_id'].nunique()} drones")

folds = list(get_folds(tab, n_splits=5, seed=RANDOM_STATE))
for tr_idx, val_idx in folds:
    assert_disjoint_groups(tab, tr_idx, val_idx)
print(f"{len(folds)} group-disjoint folds ready")


Train: 15872 flights, 620 drones, base rate = 0.1251


Test: 7534 flights, 290 drones
5 group-disjoint folds ready


## 2. Stage 1 — per-modality baselines

What each modality is worth alone, before any fusion. This is the number
fusion has to clear to prove the modalities actually interact.


In [11]:
# --- Tabular ---
X_tab, tab_cat_cols = build_tabular_features(tab)
tab_feature_cols = [c for c in X_tab.columns if c != "flight_id"]

tab_oof = np.zeros(len(tab))
tab_fold_scores = []
for tr_idx, val_idx in folds:
    model = lgb.LGBMClassifier(**LGB_PARAMS)
    model.fit(X_tab.loc[tr_idx, tab_feature_cols], y[tr_idx], categorical_feature=tab_cat_cols)
    val_pred = model.predict_proba(X_tab.loc[val_idx, tab_feature_cols])[:, 1]
    tab_oof[val_idx] = val_pred
    tab_fold_scores.append(auprc(y[val_idx], val_pred))
tab_results = report_folds(tab_fold_scores, "Tabular")


Tabular: fold AUPRCs = [0.2941, 0.2831, 0.2759, 0.3152, 0.2995]
Tabular: mean=0.2936  std=0.0136  (floor~0.125)


In [12]:
# --- Signal (hand-engineered spectral/statistical features) ---
X_sig = build_signal_features(signals, channel_names)
sig_feature_cols = X_sig.columns.tolist()

sig_oof = np.zeros(len(tab))
sig_fold_scores = []
for tr_idx, val_idx in folds:
    model = lgb.LGBMClassifier(**LGB_PARAMS)
    model.fit(X_sig.loc[tr_idx, sig_feature_cols], y[tr_idx])
    val_pred = model.predict_proba(X_sig.loc[val_idx, sig_feature_cols])[:, 1]
    sig_oof[val_idx] = val_pred
    sig_fold_scores.append(auprc(y[val_idx], val_pred))
sig_results = report_folds(sig_fold_scores, "Signal")


Signal: fold AUPRCs = [0.2981, 0.2788, 0.2795, 0.2839, 0.3037]
Signal: mean=0.2888  std=0.0102  (floor~0.125)


In [13]:
# --- Text: TF-IDF (fit inside each fold) ---
miss = missing_indicator(note_text).reshape(-1, 1)
tfidf_oof = np.zeros(len(tab))
tfidf_fold_scores = []
for tr_idx, val_idx in folds:
    tr_X, val_X = tfidf_features(note_text.iloc[tr_idx], note_text.iloc[val_idx])
    tr_X, val_X = hstack([tr_X, miss[tr_idx]]), hstack([val_X, miss[val_idx]])
    clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    clf.fit(tr_X, y[tr_idx])
    val_pred = clf.predict_proba(val_X)[:, 1]
    tfidf_oof[val_idx] = val_pred
    tfidf_fold_scores.append(auprc(y[val_idx], val_pred))
tfidf_results = report_folds(tfidf_fold_scores, "Text (TF-IDF)")


Text (TF-IDF): fold AUPRCs = [0.4394, 0.4106, 0.4207, 0.4178, 0.4261]
Text (TF-IDF): mean=0.4229  std=0.0096  (floor~0.125)


In [14]:
# --- Text: frozen sentence embeddings (only 24 unique note templates in the whole dataset) ---
unique_texts = note_text.unique()
emb_lookup = dict(zip(unique_texts, embed_notes(unique_texts.tolist())))
note_emb = np.vstack(note_text.map(emb_lookup).values)

emb_oof = np.zeros(len(tab))
emb_fold_scores = []
for tr_idx, val_idx in folds:
    tr_X = np.hstack([note_emb[tr_idx], miss[tr_idx]])
    val_X = np.hstack([note_emb[val_idx], miss[val_idx]])
    clf = LogisticRegression(max_iter=2000, class_weight="balanced")
    clf.fit(tr_X, y[tr_idx])
    val_pred = clf.predict_proba(val_X)[:, 1]
    emb_oof[val_idx] = val_pred
    emb_fold_scores.append(auprc(y[val_idx], val_pred))
emb_results = report_folds(emb_fold_scores, "Text (BGE embeddings)")


C:\Users\aneja\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Text (BGE embeddings): fold AUPRCs = [0.4399, 0.4084, 0.4253, 0.4159, 0.4261]
Text (BGE embeddings): mean=0.4231  std=0.0106  (floor~0.125)


In [15]:
stage1_summary = pd.DataFrame([
    {"modality": "Tabular", "mean_AUPRC": tab_results["mean"], "std": tab_results["std"]},
    {"modality": "Signal (hand-engineered)", "mean_AUPRC": sig_results["mean"], "std": sig_results["std"]},
    {"modality": "Text (TF-IDF)", "mean_AUPRC": tfidf_results["mean"], "std": tfidf_results["std"]},
    {"modality": "Text (BGE embeddings)", "mean_AUPRC": emb_results["mean"], "std": emb_results["std"]},
]).set_index("modality")
stage1_summary


,mean_AUPRC,std
modality,,
Tabular,0.293566,0.013593
Signal (hand-engineered),0.288801,0.010202
Text (TF-IDF),0.422911,0.009633
Text (BGE embeddings),0.423146,0.010598


## 3. Stage 2 — missingness handling

Build the combined tabular+signal+text-embedding feature panel that every
later stage reuses, then prove modality-dropout training augmentation
actually changes model behavior under simulated missingness.


In [16]:
panel_clean, cat_cols, text_cols = build_feature_panel(tab, note_text, signals, channel_names)
feature_cols = [c for c in panel_clean.columns if c != "flight_id"]
panel_test, _, _ = build_feature_panel(tab_test, note_text_test, signals_test, channel_names)
print("train panel:", panel_clean.shape, " test panel:", panel_test.shape)

# globally-degraded panels, used only for *evaluating* robustness, never for training
all_blank_text = pd.Series([""] * len(note_text), index=note_text.index)
panel_text_blank, _, _ = build_feature_panel(tab, all_blank_text, signals, channel_names)
all_blank_signal = np.zeros_like(signals)
panel_signal_blank, _, _ = build_feature_panel(tab, note_text, all_blank_signal, channel_names)


train panel: (15872, 1181)  test panel: (7534, 1181)


In [17]:
ablation_rows = []
for fold_i, (tr_idx, val_idx) in enumerate(folds):
    model_a = lgb.LGBMClassifier(**LGB_PARAMS)
    model_a.fit(panel_clean.loc[tr_idx, feature_cols], y[tr_idx], categorical_feature=cat_cols)

    sig_aug, txt_aug, _, _ = apply_modality_dropout(signals, note_text, tr_idx, p_signal=0.15, p_text=0.15, seed=RANDOM_STATE + fold_i)
    panel_aug, _, _ = build_feature_panel(tab, txt_aug, sig_aug, channel_names)
    model_b = lgb.LGBMClassifier(**LGB_PARAMS)
    model_b.fit(panel_aug.loc[tr_idx, feature_cols], y[tr_idx], categorical_feature=cat_cols)

    for name, model in [("A (no dropout aug)", model_a), ("B (with dropout aug)", model_b)]:
        normal = auprc(y[val_idx], model.predict_proba(panel_clean.loc[val_idx, feature_cols])[:, 1])
        text_missing = auprc(y[val_idx], model.predict_proba(panel_text_blank.loc[val_idx, feature_cols])[:, 1])
        signal_missing = auprc(y[val_idx], model.predict_proba(panel_signal_blank.loc[val_idx, feature_cols])[:, 1])
        ablation_rows.append({"fold": fold_i, "model": name, "normal": normal, "text_missing": text_missing, "signal_missing": signal_missing})

ablation = pd.DataFrame(ablation_rows)
ablation.groupby("model")[["normal", "text_missing", "signal_missing"]].mean()


,normal,text_missing,signal_missing
model,,,
A (no dropout aug),0.522395,0.308319,0.491025
B (with dropout aug),0.518727,0.306357,0.519057


## 4. Stage 3 — mid-level fusion

One LightGBM model on the concatenated tabular+signal+text panel, trained
with the modality-dropout augmentation from Stage 2 on each fold's training
rows. This is the model every fusion-stage number is compared against.


In [18]:
oof_fusion = np.zeros(len(tab))
test_preds_fusion_folds = []
fusion_fold_scores = []

for fold_i, (tr_idx, val_idx) in enumerate(folds):
    sig_aug, txt_aug, _, _ = apply_modality_dropout(signals, note_text, tr_idx, p_signal=0.15, p_text=0.15, seed=RANDOM_STATE + fold_i)
    panel_aug, _, _ = build_feature_panel(tab, txt_aug, sig_aug, channel_names)

    model = lgb.LGBMClassifier(**LGB_PARAMS)
    model.fit(panel_aug.loc[tr_idx, feature_cols], y[tr_idx], categorical_feature=cat_cols)

    val_pred = model.predict_proba(panel_clean.loc[val_idx, feature_cols])[:, 1]
    oof_fusion[val_idx] = val_pred
    fusion_fold_scores.append(auprc(y[val_idx], val_pred))
    test_preds_fusion_folds.append(model.predict_proba(panel_test[feature_cols])[:, 1])

fusion_results = report_folds(fusion_fold_scores, "Mid-fusion")
test_preds_fusion = np.mean(test_preds_fusion_folds, axis=0)


Mid-fusion: fold AUPRCs = [0.5254, 0.5018, 0.507, 0.5343, 0.5252]
Mid-fusion: mean=0.5187  std=0.0123  (floor~0.125)


## 5. Stage 4 — multi-task auxiliary head (`failure_mode`)

`failure_mode` (bearing/battery/none) is train-only, so it can't be a model
*input* -- but it can be an auxiliary *target*. Reformulated here as a joint
6-class problem over `(failure_within_horizon, failure_mode)`, marginalizing
back to `P(failure_within_horizon=1)` at inference by summing the three
"fails" classes. Same CV folds, same dropout augmentation, same panel, so
this is a fair apples-to-apples comparison against Stage 3.


In [19]:
def to_joint_class(y_val, m_val):
    base = {"none": 0, "battery": 1, "bearing": 2}[m_val]
    return base + (3 if y_val == 1 else 0)

joint_y = np.array([to_joint_class(yv, mv) for yv, mv in zip(y, m)])
pd.Series(joint_y).value_counts().sort_index()


0    13817
1       23
2       46
3      575
4      452
5      959
Name: count, dtype: int64

In [20]:
MULTITASK_PARAMS = {**LGB_PARAMS, "objective": "multiclass", "num_class": 6}

oof_multitask = np.zeros(len(tab))
test_preds_multitask_folds = []
multitask_fold_scores = []

for fold_i, (tr_idx, val_idx) in enumerate(folds):
    sig_aug, txt_aug, _, _ = apply_modality_dropout(signals, note_text, tr_idx, p_signal=0.15, p_text=0.15, seed=RANDOM_STATE + fold_i)
    panel_aug, _, _ = build_feature_panel(tab, txt_aug, sig_aug, channel_names)

    model = lgb.LGBMClassifier(**MULTITASK_PARAMS)
    model.fit(panel_aug.loc[tr_idx, feature_cols], joint_y[tr_idx], categorical_feature=cat_cols)

    val_probs = model.predict_proba(panel_clean.loc[val_idx, feature_cols])
    val_pred = val_probs[:, 3] + val_probs[:, 4] + val_probs[:, 5]
    oof_multitask[val_idx] = val_pred
    multitask_fold_scores.append(auprc(y[val_idx], val_pred))

    test_probs = model.predict_proba(panel_test[feature_cols])
    test_preds_multitask_folds.append(test_probs[:, 3] + test_probs[:, 4] + test_probs[:, 5])

multitask_results = report_folds(multitask_fold_scores, "Multi-task (6-class)")
test_preds_multitask = np.mean(test_preds_multitask_folds, axis=0)


Multi-task (6-class): fold AUPRCs = [0.512, 0.5035, 0.4973, 0.5293, 0.5233]
Multi-task (6-class): mean=0.5131  std=0.0119  (floor~0.125)


In [21]:
stage34_summary = pd.DataFrame([
    {"model": "Stage 3: mid-fusion", "mean_AUPRC": fusion_results["mean"], "std": fusion_results["std"], "pooled_OOF": auprc(y, oof_fusion)},
    {"model": "Stage 4: multi-task", "mean_AUPRC": multitask_results["mean"], "std": multitask_results["std"], "pooled_OOF": auprc(y, oof_multitask)},
]).set_index("model")
stage34_summary


,mean_AUPRC,std,pooled_OOF
model,,,
Stage 3: mid-fusion,0.518727,0.012273,0.516346
Stage 4: multi-task,0.513074,0.011927,0.512357


## 6. Stage 5 — model selection and calibration

**Model selection first.** Stage 4's auxiliary head was meant to regularize
the representation, but it scores *lower* than plain Stage 3 fusion on
validated AUPRC, consistently across repeated runs (~0.005-0.006 mean AUPRC
below Stage 3, bigger than run-to-run noise). Since AUPRC is the literal
competition metric, the final model is **Stage 3's plain fusion**, not the
multi-task version -- the auxiliary-head experiment was worth running and
worth keeping in this notebook as a documented negative result, but it isn't
what gets calibrated and submitted.

**Then calibration**, fit on Stage 3's pooled out-of-fold predictions.
Isotonic regression improves Brier score the most but introduces flat-bin
ties that cost a little ranking resolution; Platt scaling (a monotonic
sigmoid) can't lose any AUPRC by construction, since it never reorders
predictions. We pick whichever keeps AUPRC intact, since that's the
competition metric.


In [22]:
brier_raw = brier_score_loss(y, oof_fusion)
auprc_raw = auprc(y, oof_fusion)

isotonic = IsotonicRegression(out_of_bounds="clip")
isotonic.fit(oof_fusion, y)
cal_oof_iso = isotonic.predict(oof_fusion)

platt = LogisticRegression(penalty=None, solver="lbfgs")
platt.fit(oof_fusion.reshape(-1, 1), y)
cal_oof_platt = platt.predict_proba(oof_fusion.reshape(-1, 1))[:, 1]

calibration_summary = pd.DataFrame([
    {"calibrator": "raw (uncalibrated)", "brier": brier_raw, "AUPRC": auprc_raw},
    {"calibrator": "isotonic", "brier": brier_score_loss(y, cal_oof_iso), "AUPRC": auprc(y, cal_oof_iso)},
    {"calibrator": "platt", "brier": brier_score_loss(y, cal_oof_platt), "AUPRC": auprc(y, cal_oof_platt)},
]).set_index("calibrator")
calibration_summary


C:\Users\aneja\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


,brier,AUPRC
calibrator,,
raw (uncalibrated),0.082648,0.516346
isotonic,0.081096,0.504483
platt,0.083576,0.516346


## 7. Final submission

In [23]:
final_test_preds = platt.predict_proba(test_preds_fusion.reshape(-1, 1))[:, 1]

submission = pd.DataFrame({
    "flight_id": tab_test["flight_id"],
    "failure_within_horizon": final_test_preds,
})
submission.to_csv("../submission.csv", index=False)

sample_submission = pd.read_csv("../data/sample_submission.csv")
assert len(submission) == len(sample_submission)
assert set(submission["flight_id"]) == set(sample_submission["flight_id"])
assert submission["failure_within_horizon"].between(0, 1).all()
assert not submission["failure_within_horizon"].isna().any()
print("submission.csv written and verified:", submission.shape)
submission["failure_within_horizon"].describe()


submission.csv written and verified: (7534, 2)


count    7534.000000
mean        0.127532
std         0.163354
min         0.056516
25%         0.064007
50%         0.070093
75%         0.100391
max         0.963105
Name: failure_within_horizon, dtype: float64